In [ ]:
# If there's a conflict run this command first

# !pip uninstall -y chromadb

In [1]:
%pip install chromadb langchain-core langchain-community langchain-text-splitters nest_asyncio pinecone-client sentence-transformers scikit-learn tf-keras

Note: you may need to restart the kernel to use updated packages.


In [2]:
import posthog, deepeval, chromadb
print("posthog", posthog.__version__)
print("deepeval", deepeval.__version__)
print("chromadb", chromadb.__version__)

posthog 5.4.0
deepeval 3.6.9
chromadb 1.3.4


In [3]:

import asyncio
import json
import random
import time
from pathlib import Path
import sys

import nest_asyncio
from openai import RateLimitError

NOTEBOOK_DIR = Path.cwd().resolve()
CHAT_UTILS_DIR = NOTEBOOK_DIR.parent
if str(CHAT_UTILS_DIR) not in sys.path:
    sys.path.append(str(CHAT_UTILS_DIR))

nest_asyncio.apply()

from rag_pipeline import rag_search
import local_vectorstore

SNAPSHOT_PATH = Path('test_apartments.json')
OUTPUT_PATH = Path('goldens_from_rag.jsonl')

NUM_QUERIES = 25
TOP_K = 5
PER_QUERY_DELAY = 2.5  # seconds between calls to stay under rate limits
MAX_RETRIES = 3

local_vectorstore.initialize_local_store(SNAPSHOT_PATH)
LOCAL_SEARCH_FN = local_vectorstore.parallel_hybrid_search
print('Local vector store ready.')


/opt/anaconda3/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid compatibility violations i

Local vector store ready.


In [4]:

with SNAPSHOT_PATH.open() as f:
    APARTMENTS = json.load(f)

random.seed(42)
random.shuffle(APARTMENTS)

NEIGHBOR_FALLBACK = 'manhattan'
AMENITY_FALLBACKS = ['laundry', 'doorman', 'gym', 'dishwasher']

def pick_amenities(record, k=2):
    amenities = record.get('amenities') or []
    cleaned = [a.replace('_', ' ') for a in amenities]
    cleaned = cleaned or AMENITY_FALLBACKS
    random.shuffle(cleaned)
    return cleaned[:k]

def build_query_from_listing(record):
    beds = record.get('bedrooms') or 1
    baths = record.get('bathrooms') or 1
    neighborhood = (record.get('neighborhood') or NEIGHBOR_FALLBACK).replace('-', ' ')
    borough = (record.get('borough') or 'manhattan').title()
    price = int(record.get('price') or 3000)
    budget = int(price * 1.05 // 50 * 50)
    amenities = pick_amenities(record)
    amenity_text = ' and '.join(amenities)
    return (
        f"Looking for a {int(beds)} bedroom, {int(baths)} bath place in {neighborhood} "
        f"{borough} under ${budget:,} with {amenity_text}."
    )

def sample_queries(n):
    queries = []
    for rec in APARTMENTS:
        query = build_query_from_listing(rec)
        queries.append({'query': query, 'listing_id': rec.get('id')})
        if len(queries) >= n:
            break
    return queries

seed_queries = sample_queries(NUM_QUERIES)
print(f'Prepared {len(seed_queries)} seed queries.')
for example in seed_queries[:5]:
    print('-', example['query'])


Prepared 25 seed queries.
- Looking for a 2 bedroom, 2 bath place in murray hill Manhattan under $8,300 with gym and roofdeck.
- Looking for a 1 bedroom, 1 bath place in chelsea Manhattan under $5,900 with patio and doorman.
- Looking for a 1 bedroom, 1 bath place in morningside heights Manhattan under $5,200 with live in super and pets.
- Looking for a 3 bedroom, 1 bath place in chelsea Manhattan under $6,800 with dishwasher and pets.
- Looking for a 1 bedroom, 1 bath place in hells kitchen Manhattan under $4,900 with courtyard and doorman.


In [5]:

def serialize_match(match):
    md = match.metadata
    return {
        'listing_id': md.get('listing_id'),
        'price': md.get('price'),
        'bedrooms': md.get('bedrooms'),
        'bathrooms': md.get('bathrooms'),
        'neighborhood': md.get('neighborhood'),
        'borough': md.get('borough'),
        'amenities': md.get('amenities'),
        'score': match.score,
    }


async def run_single_turn(query_text):
    attempt = 0
    while True:
        try:
            return await rag_search(
                user_query=query_text,
                top_k=TOP_K,
                chat_history=[],
                is_first_turn=True,
                return_full_context=True,
                search_fn=LOCAL_SEARCH_FN,
            )
        except RateLimitError as err:
            attempt += 1
            if attempt > MAX_RETRIES:
                raise
            wait = PER_QUERY_DELAY * (attempt + 1)
            print(f"Rate limited; retrying in {wait:.1f}s... ({err})")
            await asyncio.sleep(wait)
        except Exception as err:
            attempt += 1
            if attempt > MAX_RETRIES:
                raise
            wait = 3 * attempt
            print(f"Error '{err}' — retrying in {wait}s")
            await asyncio.sleep(wait)


async def build_goldens():
    goldens = []
    for idx, payload in enumerate(seed_queries, start=1):
        query = payload['query']
        print(f"[{idx}/{len(seed_queries)}] {query}")
        result = await run_single_turn(query)
        listing_ids = [m.metadata.get('listing_id') for m in result['matches']]
        golden = {
            'input': result['current_user_prompt'],
            'expected_output': result['llm_output'],
            'clarification': result['clarification'],
            'listing_ids': listing_ids,
            'context_block': result['context_block'],
            'standalone_query': result['standalone_query'],
            'serialized_listings': [serialize_match(m) for m in result['matches']],
        }
        goldens.append(golden)
        if PER_QUERY_DELAY:
            await asyncio.sleep(PER_QUERY_DELAY)
    return goldens


goldens = asyncio.run(build_goldens())
print(f"Generated {len(goldens)} goldens.")


Task exception was never retrieved
future: <Task finished name='Task-2' coro=<BaseApiClient.aclose() done, defined at /opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py:1812> exception=AttributeError("'BaseApiClient' object has no attribute '_async_httpx_client'")>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py", line 1815, in aclose
    await self._async_httpx_client.aclose()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'BaseApiClient' object has no attribute '_async_httpx_client'
Task exception was never retrieved
future: <Task finished name='Task-6' coro=<BaseApiClient.aclose() done, defined at /opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py:1812> exception=AttributeError("'BaseApiClient' object has no attribute

[1/25] Looking for a 2 bedroom, 2 bath place in murray hill Manhattan under $8,300 with gym and roofdeck.
[WARN] Pinecone filter extraction failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Amenity parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Neighborhood parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Subway parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.

Task exception was never retrieved
future: <Task finished name='Task-8' coro=<BaseApiClient.aclose() done, defined at /opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py:1812> exception=AttributeError("'BaseApiClient' object has no attribute '_async_httpx_client'")>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py", line 1815, in aclose
    await self._async_httpx_client.aclose()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'BaseApiClient' object has no attribute '_async_httpx_client'


Soft Filters: {'amenities': {'$in': ['garden', 'recreation_facilities', 'roofdeck', 'gym', 'patio', 'terrace', 'private_roof_deck', 'balcony', 'deck']}, 'neighborhood': {'$in': ['murray-hill']}}
Retrieved 15 unique matches total.


Task exception was never retrieved
future: <Task finished name='Task-9' coro=<BaseApiClient.aclose() done, defined at /opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py:1812> exception=AttributeError("'BaseApiClient' object has no attribute '_async_httpx_client'")>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py", line 1815, in aclose
    await self._async_httpx_client.aclose()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'BaseApiClient' object has no attribute '_async_httpx_client'
Task exception was never retrieved
future: <Task finished name='Task-13' coro=<BaseApiClient.aclose() done, defined at /opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py:1812> exception=AttributeError("'BaseApiClient' object has no attribut

...

[23/25] Looking for a 2 bedroom, 1 bath place in hamilton heights Manhattan under $3,000 with hardwood floors and city view.
[WARN] Pinecone filter extraction failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Amenity parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Neighborhood parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Subway parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `lo

Task exception was never retrieved
future: <Task finished name='Task-163' coro=<BaseApiClient.aclose() done, defined at /opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py:1812> exception=AttributeError("'BaseApiClient' object has no attribute '_async_httpx_client'")>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py", line 1815, in aclose
    await self._async_httpx_client.aclose()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'BaseApiClient' object has no attribute '_async_httpx_client'
Task exception was never retrieved
future: <Task finished name='Task-167' coro=<BaseApiClient.aclose() done, defined at /opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py:1812> exception=AttributeError("'BaseApiClient' object has no attri

[24/25] Looking for a 1 bedroom, 1 bath place in midtown Manhattan under $4,050 with storage room and live in super.
[WARN] Pinecone filter extraction failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Amenity parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Neighborhood parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Subway parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`)

Task exception was never retrieved
future: <Task finished name='Task-170' coro=<BaseApiClient.aclose() done, defined at /opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py:1812> exception=AttributeError("'BaseApiClient' object has no attribute '_async_httpx_client'")>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py", line 1815, in aclose
    await self._async_httpx_client.aclose()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'BaseApiClient' object has no attribute '_async_httpx_client'
Task exception was never retrieved
future: <Task finished name='Task-174' coro=<BaseApiClient.aclose() done, defined at /opt/anaconda3/lib/python3.12/site-packages/google/genai/_api_client.py:1812> exception=AttributeError("'BaseApiClient' object has no attri

[25/25] Looking for a 1 bedroom, 1 bath place in east village Manhattan under $4,700 with washer dryer and storage room.
[WARN] Pinecone filter extraction failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Amenity parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Neighborhood parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.. Falling back to regex.
[WARN] Subway parsing failed: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `locati

In [6]:
import datetime

timestamp = datetime.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
output_file = OUTPUT_PATH.with_name(f"goldens_{timestamp}.jsonl")

with output_file.open("w") as f:
    for record in goldens:
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(goldens)} records to {output_file}")
output_file


Saved 25 records to goldens_20251116-215703.jsonl


/var/folders/y4/kt57fxd942z3zncr4rvv6t740000gn/T/ipykernel_97940/3509147569.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.datetime.utcnow().strftime("%Y%m%d-%H%M%S")


PosixPath('goldens_20251116-215703.jsonl')

In [11]:
import csv

# Define the CSV output path
csv_output_file = output_file.with_suffix('.csv')

# Read the JSONL and write to CSV
with output_file.open('r') as fin, csv_output_file.open('w', newline='', encoding='utf-8') as fout:
    writer = None
    for line in fin:
        record = json.loads(line)
        # Flatten fields as needed for CSV
        row = {
            'input': record.get('input', ''),
            'expected_output': record.get('expected_output', ''),
            'clarification': record.get('clarification', ''),
            'listing_ids': ','.join(record.get('listing_ids', [])),
            'standalone_query': record.get('standalone_query', ''),
        }
        if writer is None:
            writer = csv.DictWriter(fout, fieldnames=row.keys())
            writer.writeheader()
        writer.writerow(row)

print(f"Saved CSV to {csv_output_file}")
csv_output_file

Saved CSV to goldens_20251116-215703.csv


PosixPath('goldens_20251116-215703.csv')

ImportError: cannot import name 'RetrievalMetric' from 'deepeval' (/opt/anaconda3/lib/python3.12/site-packages/deepeval/__init__.py)